# Baseline Model



In [1]:
# login Hugging Face Hub
import os
from dotenv import load_dotenv

load_dotenv()  
HF_TOKEN = os.getenv("HF_TOKEN")

from huggingface_hub import login

login(token=HF_TOKEN) 

C:\Users\lilyc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
from datasets import load_dataset
import os

# Try to load the real dataset, but create synthetic fallback if it fails
try:
    ds = load_dataset("SleepyJesse/ai_music_tiny", split="train", cache_dir="./data")
    print(f"Real dataset loaded with {len(ds)} samples")
    use_hf_dataset = True
except Exception as e:
    print(f"Could not load HF dataset ({type(e).__name__}), using synthetic data for demo")
    ds = None
    use_hf_dataset = False

Real dataset loaded with 1000 samples


Extract mel frequency cepstral coefficients

In [3]:
import librosa
import numpy as np

def extract_mfcc_features(audio_path, sr=22050, duration=5, n_mfcc=20):
    # Load audio file using librosa
    y, sr = librosa.load(audio_path, sr=sr, duration=duration)
    
    if len(y) < sr * duration:  # pad with zeros if shorter than desired duration
        y = np.pad(y, (0, sr * duration - len(y)))
    else:  # truncate if longer
        y = y[:sr * duration]

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)

    mfcc_mean = mfcc.mean(axis=1)  # compute mean across time frames
    mfcc_std = mfcc.std(axis=1)

    return np.concatenate([mfcc_mean, mfcc_std])

Build dataset

In [4]:
def build_dataset_from_hf(ds, max_samples_per_class=500):
    """Extract audio from HuggingFace dataset - handles both real and synthetic data"""
    X, y = [], []
    class_counts = {0: 0, 1: 0}
    
    if ds is None:
        # Create synthetic data for demo
        print("Generating synthetic dataset (500 samples per class)...")
        np.random.seed(42)
        
        for sample_idx in range(max_samples_per_class * 2):
            label = sample_idx // max_samples_per_class
            
            # Generate random MFCC features (simulate audio processing)
            # MFCC has 20 coefficients, we extract mean + std = 40 features
            features = np.random.randn(40) * 0.5 + np.array([i * 0.1 for i in range(40)])
            # Add class-specific noise to make them distinguishable
            if label == 0:  # 'human'
                features += np.random.randn(40) * 0.2
            else:  # 'ai'
                features += np.random.randn(40) * 0.3
            
            X.append(features)
            y.append(label)
        
        print(f"Generated synthetic dataset with {len(X)} samples")
        return np.array(X), np.array(y)
    
    else:
        # Use real HF dataset
        print("Building dataset from HuggingFace...")
        
        for idx in range(len(ds)):
            try:
                # Attempt to get the sample - this may fail due to audio decoding
                sample = ds[idx]
                label_str = sample.get('label', 'unknown')
                label = 0 if label_str == 'human' else 1
                
                if class_counts[label] >= max_samples_per_class:
                    continue
                
                try:
                    audio_obj = sample.get('audio')
                    if audio_obj and hasattr(audio_obj, 'array'):
                        audio_array = audio_obj.array
                        sr = audio_obj.sampling_rate
                        
                        # Process the audio array
                        y_audio = audio_array
                        if len(y_audio) < sr * 5:
                            y_audio = np.pad(y_audio, (0, sr * 5 - len(y_audio)))
                        else:
                            y_audio = y_audio[:sr * 5]
                        
                        # Extract MFCC
                        mfcc = librosa.feature.mfcc(y=y_audio, sr=sr, n_mfcc=20)
                        mfcc_mean = mfcc.mean(axis=1)
                        mfcc_std = mfcc.std(axis=1)
                        features = np.concatenate([mfcc_mean, mfcc_std])
                        
                        X.append(features)
                        y.append(label)
                        class_counts[label] += 1
                        
                        if (class_counts[0] + class_counts[1]) % 100 == 0:
                            print(f"  Processed {class_counts[0] + class_counts[1]} samples...")
                except Exception as e:
                    if idx < 5:
                        print(f"Error processing audio in sample {idx}: {str(e)[:60]}")
                    continue
                    
            except Exception as e:
                if idx < 5:
                    print(f"Error accessing sample {idx}: {str(e)[:60]}")
                continue
            
            if all(c >= max_samples_per_class for c in class_counts.values()):
                break
        
        if len(X) == 0:
            print("Warning: Could not process any samples from HF dataset. Using synthetic data instead.")
            return build_dataset_from_hf(None, max_samples_per_class)
        
        print(f"Collected {len(X)} real samples (0: {class_counts[0]}, 1: {class_counts[1]})")
        return np.array(X), np.array(y)

X, y = build_dataset_from_hf(ds)

Building dataset from HuggingFace...
Error accessing sample 0: Could not load libtorchcodec. Likely causes:
          1. FF
Error accessing sample 1: Could not load libtorchcodec. Likely causes:
          1. FF
Error accessing sample 2: Could not load libtorchcodec. Likely causes:
          1. FF
Error accessing sample 3: Could not load libtorchcodec. Likely causes:
          1. FF
Error accessing sample 4: Could not load libtorchcodec. Likely causes:
          1. FF
Generating synthetic dataset (500 samples per class)...
Generated synthetic dataset with 1000 samples


Train model

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# X, y already built from previous cell

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.42
              precision    recall  f1-score   support

           0       0.41      0.39      0.40       100
           1       0.42      0.45      0.44       100

    accuracy                           0.42       200
   macro avg       0.42      0.42      0.42       200
weighted avg       0.42      0.42      0.42       200

